# 08 Managed Agents — 互動式學習 Lab

本 notebook 整合 `08_managed_agents` 所有概念，提供可直接執行的學習環境。

## 學習路線
1. Managed Agents 概念速覽
2. Agent + Session 建立
3. Custom Tools 定義與測試
4. Memory Store CRUD
5. SSE Event Stream 處理
6. 材料科學 Extraction 完整 Demo
7. 小專案：設計你自己的 materials agent

> ⚠️ **Beta Access**: 本 notebook 預設使用 mock 模式。有 beta access 時設定 `LIVE_MODE = True`。

In [ ]:
import sys
import os
import json

# 加入 08_managed_agents 到 path，可以 import 各模組的函式
module_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

LIVE_MODE = False  # 改成 True 需要 beta access + 正確 SDK

print(f'模式: {"LIVE" if LIVE_MODE else "MOCK"}')
print('準備就緒！')

---
## 1. Managed Agents 概念速覽

```
┌─────────────────────────────────────────────────────┐
│  Agent Definition                                   │
│  ├── Model: claude-sonnet-4-6                       │
│  ├── System Prompt                                  │
│  ├── Built-in Tools: bash, web_search, files        │
│  └── Custom Tools: your API calls                   │
│                                                     │
│  Environment: Python 3.11 + packages                │
│                                                     │
│  Session = Agent + Environment + Memory Stores      │
│  └── Messages (user ↔ agent) + SSE event stream     │
│                                                     │
│  Memory Store: key-value + version history          │
└─────────────────────────────────────────────────────┘
```

### 關鍵差異 vs 直接用 API
| 直接用 Anthropic API | Managed Agents |
|---------------------|----------------|
| 自管對話歷史 | Session 自動管理 |
| 自管工具執行環境 | 雲端 Environment |
| 自管 state | Memory Store |
| 手動 streaming | SSE 原生支援 |

---
## 2. Agent + Session 建立

In [ ]:
# 複用 01_quickstart.py 的 Mock Client
from quickstart import MockManagedAgentsClient

client = MockManagedAgentsClient()

# 建立 agent
agent = client.create_agent(
    name='materials-agent',
    model='claude-sonnet-4-6',
    system_prompt='你是材料科學文獻分析專家',
    tools=['bash', 'web_search'],
)
print(f'Agent: {agent["id"]}')

# 建立 environment
env = client.create_environment(
    name='materials-env',
    runtime='python3.11',
    packages=['numpy', 'pandas'],
)
print(f'Environment: {env["id"]}')

# 建立 session
session = client.create_session(
    agent_id=agent['id'],
    environment_id=env['id'],
)
print(f'Session: {session["id"]}')

---
## 3. Custom Tools 定義與測試

In [ ]:
from custom_tools import lookup_material_property, convert_units, handle_tool_call

# 測試 tool
result = lookup_material_property('MAPbI3', 'band_gap')
print('lookup_material_property(MAPbI3, band_gap):')
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 單位換算
result = convert_units(1.57, 'eV', 'kJ/mol')
print('convert_units(1.57, eV, kJ/mol):')
print(json.dumps(result, ensure_ascii=False, indent=2))

### 練習 3.1
定義一個新的 custom tool `calculate_bandgap_wavelength`，計算 band gap 對應的光子波長（nm）。

> 提示：E = hc/λ，E(eV) = 1240 / λ(nm)

In [ ]:
# TODO: 在這裡定義你的 tool
my_tool_schema = {
    'name': 'calculate_bandgap_wavelength',
    'description': '...',  # 填入 description
    'input_schema': {
        'type': 'object',
        'properties': {
            # 填入 properties
        },
        'required': [],
    },
}

def calculate_bandgap_wavelength(band_gap_eV: float) -> dict:
    # TODO: 實作換算邏輯
    pass

# 測試
# result = calculate_bandgap_wavelength(1.57)
# print(result)  # 應輸出約 790 nm

---
## 4. Memory Store CRUD

In [ ]:
from memory_stores import MockMemoryStoreClient

store_client = MockMemoryStoreClient()
playbook = store_client.create('materials-playbook', '材料提取規則')

# Seed
playbook.seed({
    'rules/step1': '先看摘要和結論',
    'rules/step2': '掃描所有表格',
    'heuristics/champion': 'champion device → 最佳樣品，非平均',
})

# Read
entry = playbook.read('heuristics/champion')
print(f'v{entry["version"]}: {entry["value"]}')

In [ ]:
# Safe update
new_value = entry['value'] + '\n- best performing → 同義'
updated = playbook.write('heuristics/champion', new_value,
                          precondition_version=entry['version'])
print(f'更新後 version: {updated["version"]}')

# History
history = playbook.history('heuristics/champion')
print(f'共 {len(history)} 個版本')

### 練習 4.1
實作一個 `curator_update(store, key, delta_text)` 函數，加入 Brevity Bias 防護（如果新內容比舊內容短 20% 以上就拒絕更新）。

In [ ]:
def curator_update(store, key, delta_text, min_length_ratio=0.8):
    """
    安全的 Curator delta update。
    - Append delta_text 到現有內容
    - 防止 Brevity Bias（新內容長度不得低於舊內容的 min_length_ratio）
    - 使用 precondition_version 避免並發衝突
    """
    # TODO: 實作
    pass

# 測試
# curator_update(playbook, 'rules/step1', '\n- 同時注意 SI 補充材料')
# curator_update(playbook, 'rules/step1', 'x')  # 應失敗（太短）

---
## 5. SSE Event Stream 處理

In [ ]:
from event_streaming import StreamEventHandler, mock_extraction_event_stream

paper_abstract = 'MAPbI3 perovskite solar cell with PCE 22.7% and band gap 1.57 eV'

handler = StreamEventHandler(verbose=True)
result = handler.process_stream(mock_extraction_event_stream(paper_abstract))

print(f'\nTool calls: {len(result["tool_calls"])}')
print(f'Memory ops: {len(result["memory_ops"])}')

### 練習 5.1
繼承 `StreamEventHandler`，建立一個 `LoggingHandler`：
- 將所有 memory_write 事件記錄到 `self.write_log` list
- 在 `message_stop` 時印出統計摘要（幾次 tool_use、幾次 memory_write）

In [ ]:
class LoggingHandler(StreamEventHandler):
    def __init__(self):
        super().__init__(verbose=False)  # 靜默模式
        self.write_log = []

    def on_memory_write(self, event):
        # TODO: 記錄 write 事件
        pass

    def on_message_stop(self, event):
        # TODO: 印出統計
        pass

# 測試
# logging_handler = LoggingHandler()
# logging_handler.process_stream(mock_extraction_event_stream(paper_abstract))

---
## 6. 材料科學 Extraction 完整 Demo

In [ ]:
from material_extraction_demo import (
    MOCK_PAPER, MaterialExtractionAgent
)

mock_playbook = {
    'composition_rules': '鈣鈦礦格式：ABX3',
    'confidence_heuristics': 'high=摘要, medium=表格, low=圖表',
}
cases_store = []

agent = MaterialExtractionAgent(MOCK_PAPER, mock_playbook, cases_store)
result, report = agent.run()

print('\n=== 提取結果 (JSON) ===')
print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
print('\n=== Markdown Report ===')
print(report)

---
## 7. 小專案：設計你自己的 Materials Agent

目標：設計一個能處理**熱電材料（thermoelectric）**論文的提取 agent。

熱電材料的關鍵性質：
- **ZT**：thermoelectric figure of merit（無因次，越高越好）
- **Seebeck coefficient (S)**：單位 μV/K
- **Electrical conductivity (σ)**：單位 S/m
- **Thermal conductivity (κ)**：單位 W/(m·K)
- **測量溫度**：通常在 300–800K

### 任務
1. 定義一個 `thermoelectric_tool` custom tool
2. 設計一個 thermoelectric playbook（至少 3 條規則）
3. 建立一個 `ThermoelectricExtractionResult` dataclass
4. 實作一個簡化的 extraction agent 跑一篇 mock 論文

In [ ]:
# TODO Step 1: 定義 thermoelectric_tool schema
thermoelectric_tool = {
    'name': 'lookup_thermoelectric_property',
    'description': '查詢熱電材料的 ZT, Seebeck, 電導率, 熱導率',
    'input_schema': {
        'type': 'object',
        'properties': {
            'material': {'type': 'string'},
            'property': {
                'type': 'string',
                'enum': ['ZT', 'seebeck_coefficient', 'electrical_conductivity', 'thermal_conductivity'],
            },
            'temperature_K': {'type': 'number', 'description': '測量溫度（K）'},
        },
        'required': ['material', 'property'],
    },
}

In [ ]:
# TODO Step 2: 設計熱電材料 playbook
thermoelectric_playbook = {
    'rules/zt_location': '...',    # ZT 通常在哪裡
    'rules/temperature_range': '...',  # 溫度範圍注意事項
    'heuristics/peak_zt': '...',   # peak ZT vs average ZT
}
# TODO: 填入內容

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

# TODO Step 3: 定義 ThermoelectricExtractionResult
@dataclass
class ThermoelectricExtractionResult:
    paper_doi: str
    material: str
    # TODO: 加入 ZT, seebeck, conductivity, thermal_conductivity 等欄位
    confidence: str = 'medium'

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if v is not None}

In [ ]:
# TODO Step 4: Mock 論文 + extraction agent
MOCK_THERMOELECTRIC_PAPER = {
    'doi': '10.1039/D3EE09876B',
    'title': 'High ZT in PbTe-based thermoelectric materials',
    'abstract': '''
        We report a peak ZT of 2.2 at 773K in Bi-doped PbTe.
        Seebeck coefficient reaches 350 μV/K at 600K.
        Thermal conductivity is reduced to 0.8 W/(m·K) by nanostructuring.
    '''.strip(),
}

# TODO: 實作 extraction
# result = ThermoelectricExtractionResult(...)
# print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))

---
## 完成！

恭喜完成 08_managed_agents 模組！你已經學會：

- ✅ Managed Agents API 的核心概念（Agent, Environment, Session）
- ✅ Custom Tools 設計最佳實務（詳細 description + 清楚 schema）
- ✅ Memory Store CRUD + version history + preconditioned write
- ✅ 多 store session 設計（read-only vs read-write, shared vs scoped）
- ✅ SSE Event Stream 處理（文字、tool_use、memory events）
- ✅ 材料科學 extraction pipeline 整合
- ✅ ACE Framework 與 Managed Agents 的對應關係

### 下一步
- 申請 beta access，把 mock 換成 live API
- 將 `05_material_science_agents` 的 extraction agent 移植到 Managed Agents
- 設計完整的 GRC Loop（Generator → Reflector → Curator）用 3 個 session 實作